In [ ]:
# STEP 1
install.packages("BiocManager")

# STEP 2
options(repos = BiocManager::repositories())

# STEP 3
BiocManager::install(c(
  "DelayedArray",
  "SummarizedExperiment",
  "BiocParallel",
  "DESeq2"
), ask = FALSE, update = FALSE)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    BioCsoft: https://bioconductor.org/packages/3.23/bioc
    BioCann: https://bioconductor.org/packages/3.23/data/annotation
    BioCexp: https://bioconductor.org/packages/3.23/data/experiment
    BioCworkflows: https://bioconductor.org/packages/3.23/workflows
    BioCbooks: https://bioconductor.org/packages/3.23/books
    CRAN: https://cran.rstudio.com

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    BioCsoft: https://bioconductor.org/packages/3.23/bioc
    BioCann: https://bioconductor.org/packages/3.23/data/annotation
    BioCexp: https://bioconductor.org/packages/3.23/data/experiment
    BioCworkflows: https://bioconductor.org/package

In [ ]:
library(DESeq2)

# load data
countsS <- read.csv("/content/gse108966_esp_counts.csv", row.names=1)
labelsS <- read.csv("/content/gse108966_esp_labels.csv")

# transpose counts
countsS <- t(countsS)



In [ ]:
library(DESeq2)

# load data
counts <- read.csv("/content/gse108966_esT_counts.csv", row.names=1)
labels <- read.csv("/content/gse108966_esT_labels.csv")

# transpose counts
counts <- t(counts)



In [ ]:
# create metadata
coldata <- data.frame(
    condition = as.factor(labels$condition)
)

# build DESeq dataset
dds <- DESeqDataSetFromMatrix(
    countData = counts,
    colData = coldata,
    design = ~ condition
)

# run DESeq2
dds <- DESeq(dds)

# get results
res <- results(dds)

# sort by adjusted p-value
resOrdered <- res[order(res$padj), ]

# save results
write.csv(as.data.frame(resOrdered),
          "est_DESeq2_results.csv")

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

-- replacing outliers and refitting for 7 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)

estimating dispersions

fitting model and testing



In [ ]:
print(resOrdered)

log2 fold change (MLE): condition 1 vs 0 
Wald test p-value: condition 1 vs 0 
DataFrame with 2588 rows and 6 columns
                 baseMean log2FoldChange     lfcSE       stat      pvalue
                <numeric>      <numeric> <numeric>  <numeric>   <numeric>
hsa.miR.501.3p  609.60588      -1.514447  0.254244   -5.95667 2.57435e-09
hsa.miR.934       3.77960       2.051007  0.373419    5.49251 3.96253e-08
hsa.miR.3607.5p   2.55964       2.471624  0.471926    5.23731 1.62931e-07
hsa.miR.1827      3.74032      -2.551829  0.504135   -5.06180 4.15318e-07
hsa.miR.425.3p   79.39674      -0.752515  0.148374   -5.07175 3.94183e-07
...                   ...            ...       ...        ...         ...
hsa.miR.936     0.0000000             NA        NA         NA          NA
hsa.miR.937.5p  0.0368619       0.062960  2.082696  0.0302301    0.975884
hsa.miR.938     0.0000000             NA        NA         NA          NA
hsa.miR.9500    0.0000000             NA        NA         NA       

In [ ]:
significant <- subset(resOrdered,
                      padj < 0.05 &
                      abs(log2FoldChange) > 1)

In [ ]:
head(significant)

log2 fold change (MLE): condition 1 vs 0 
Wald test p-value: condition 1 vs 0 
DataFrame with 6 rows and 6 columns
                 baseMean log2FoldChange     lfcSE      stat      pvalue
                <numeric>      <numeric> <numeric> <numeric>   <numeric>
hsa.miR.501.3p  609.60588       -1.51445  0.254244  -5.95667 2.57435e-09
hsa.miR.934       3.77960        2.05101  0.373419   5.49251 3.96253e-08
hsa.miR.3607.5p   2.55964        2.47162  0.471926   5.23731 1.62931e-07
hsa.miR.1827      3.74032       -2.55183  0.504135  -5.06180 4.15318e-07
hsa.miR.96.5p   371.12504       -1.19449  0.234643  -5.09068 3.56776e-07
hsa.miR.335.3p  298.55079       -1.24556  0.252125  -4.94023 7.80303e-07
                       padj
                  <numeric>
hsa.miR.501.3p  2.80862e-06
hsa.miR.934     2.16156e-05
hsa.miR.3607.5p 5.92525e-05
hsa.miR.1827    7.55187e-05
hsa.miR.96.5p   7.55187e-05
hsa.miR.335.3p  9.45900e-05

In [ ]:
write.csv(as.data.frame(significant),
          "significant_deseq2_results.csv")

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
}
BiocManager::install("edgeR", update = FALSE, ask = FALSE)

library(edgeR)

# Load data for edgeR
edger_counts <- read.csv("/content/gse108966_esT_counts.csv", row.names=1)
edger_labels <- read.csv("/content/gse108966_esT_labels.csv")

# Transpose counts if necessary (edgeR expects genes as rows, samples as columns)
# Assuming the loaded counts are already in the correct orientation (genes as rows, samples as columns).
# If not, you might need to transpose: edger_counts <- t(edger_counts)
# For now, we will proceed assuming the files are structured correctly for edgeR.


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    BioCsoft: https://bioconductor.org/packages/3.23/bioc
    BioCann: https://bioconductor.org/packages/3.23/data/annotation
    BioCexp: https://bioconductor.org/packages/3.23/data/experiment
    BioCworkflows: https://bioconductor.org/packages/3.23/workflows
    BioCbooks: https://bioconductor.org/packages/3.23/books
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Installing package(s) 'edgeR'

also installing the dependencies ‘statmod’, ‘limma’


Loading required package: limma


Attaching package: ‘limma’


The following object is masked from ‘package:DESeq2’:

    plotMA


The following object is masked from ‘package:BiocGenerics’:

    plotMA




In [ ]:
library(edgeR)

# Convert to matrix
counts <- as.matrix(edger_counts)

# Replace NA
counts[is.na(counts)] <- 0

# IMPORTANT:
# transpose because edgeR expects:
# rows = genes/miRNAs
# cols = samples

counts <- t(counts)

# Ensure numeric
mode(counts) <- "numeric"

# Ensure integer counts
counts <- round(counts)

# Check dimensions
dim(counts)

# Create DGEList
dge <- DGEList(counts = counts)

# Labels
condition <- factor(edger_labels$condition)

# Check matching
length(condition)
ncol(dge)

# Must match
stopifnot(length(condition) == ncol(dge))

# Design matrix
design <- model.matrix(~condition)

# Filter low-expression miRNAs
keep <- filterByExpr(dge, design)

dge <- dge[keep,,keep.lib.sizes=FALSE]

# Normalize
dge <- calcNormFactors(dge)

# Estimate dispersion
dge <- estimateDisp(dge, design)

# Fit model
fit <- glmFit(dge, design)

# Differential expression
lrt <- glmLRT(fit)

# Results
results <- topTags(lrt, n=Inf)

# View
head(results$table)

[1] 2588  111

[1] 111

[1] 111

calcNormFactors has been renamed to normLibSizes



,logFC,logCPM,LR,PValue,FDR
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
hsa.miR.934,1.9681948,-0.4481669,44.58712,2.432873e-11,1.377006e-08
hsa.miR.345.5p,0.7852197,7.8954247,29.22196,6.454338e-08,1.826578e-05
hsa.miR.501.3p,-1.5197357,6.5272911,26.21658,3.051916e-07,5.757947e-05
hsa.miR.96.5p,-1.2962472,5.7759800,24.55046,7.238902e-07,1.024305e-04
hsa.miR.877.5p,-1.1754730,1.2631808,23.76602,1.087861e-06,1.231459e-04
hsa.miR.30d.5p,0.7047399,11.4124232,23.03757,1.588660e-06,1.497741e-04


In [ ]:
edgeR_results <- results$table

sig_mirnas <- edgeR_results[
    edgeR_results$FDR < 0.05 &
    abs(edgeR_results$logFC) > 1,
]

head(sig_mirnas)

,logFC,logCPM,LR,PValue,FDR
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
hsa.miR.934,1.968195,-0.4481669,44.58712,2.432873e-11,1.377006e-08
hsa.miR.501.3p,-1.519736,6.5272911,26.21658,3.051916e-07,5.757947e-05
hsa.miR.96.5p,-1.296247,5.7759800,24.55046,7.238902e-07,1.024305e-04
hsa.miR.877.5p,-1.175473,1.2631808,23.76602,1.087861e-06,1.231459e-04
hsa.miR.335.3p,-1.185642,5.3224232,22.74242,1.852330e-06,1.497741e-04
hsa.miR.885.5p,1.556297,0.4081597,16.85320,4.038523e-05,2.285804e-03


In [ ]:
# Get significant miRNA names from DESeq2 results
deseq2_sig_mirnas <- rownames(significant)

# Get significant miRNA names from edgeR results
edger_sig_mirnas <- rownames(sig_mirnas)

# Find the common miRNAs
common_mirnas <- intersect(deseq2_sig_mirnas, edger_sig_mirnas)

# Print the common miRNAs and their count
print(paste("Number of common significant miRNAs:", length(common_mirnas)))
if (length(common_mirnas) > 0) {
  print("Common significant miRNAs:")
  print(common_mirnas)
} else {
  print("No common significant miRNAs found between DESeq2 and edgeR at the specified thresholds.")
}

[1] "Number of common significant miRNAs: 11"
[1] "Common significant miRNAs:"
 [1] "hsa.miR.501.3p"  "hsa.miR.934"     "hsa.miR.96.5p"   "hsa.miR.335.3p" 
 [5] "hsa.miR.877.5p"  "hsa.miR.6131"    "hsa.miR.503.5p"  "hsa.miR.885.5p" 
 [9] "hsa.miR.5096"    "hsa.miR.502.5p"  "hsa.miR.193a.5p"


In [ ]:
prev1 = read("/content/EST_all_top10_consensus_RIFgse108966_Biomarkers.csv")

In [ ]:
# Re-read prev1 to ensure it's available and correctly parsed
# Assuming the miRNA names are in the first column of the CSV
prev1_data <- read.csv("/content/EST_all_top10_consensus_RIFgse108966_Biomarkers.csv")
prev_mirnas <- as.character(prev1_data[[1]])

# Find the common miRNAs between common_mirnas and prev_mirnas
common_to_all <- intersect(common_mirnas, prev_mirnas)

# Print the results
print(paste("Number of common miRNAs in DESeq2, edgeR, and prev dataset:", length(common_to_all)))
if (length(common_to_all) > 0) {
  print("Common miRNAs across all datasets:")
  print(common_to_all)
} else {
  print("No common miRNAs found across DESeq2, edgeR, and the prev dataset.")
}

[1] "Number of common miRNAs in DESeq2, edgeR, and prev dataset: 0"
[1] "No common miRNAs found across DESeq2, edgeR, and the prev dataset."
